# Analyse van onkostennota

Deze notebook toont hoe je agenten kunt maken die plugins gebruiken om reiskosten te verwerken van lokale bonnetjesafbeeldingen, een onkostennota-e-mail te genereren en onkostengegevens te visualiseren met een taartdiagram. Agenten kiezen dynamisch functies op basis van de taakcontext.

Stappen:
1. OCR Agent verwerkt de lokale bonafbeelding en haalt reiskostengegevens eruit.
2. E-mailagent genereert een onkostennota-e-mail.

### Voorbeeld van een reiskosten scenario:
Stel je bent een werknemer die voor een zakelijke vergadering naar een andere stad reist. Jouw bedrijf heeft een beleid om alle redelijke reiskosten te vergoeden. Hier is een overzicht van mogelijke reiskosten:
- Vervoer:
Vliegtickets voor een retourvlucht van jouw woonstad naar de bestemmingsstad.
Taxi- of ritdeelservices van en naar de luchthaven.
Lokaal vervoer in de bestemmingsstad (zoals openbaar vervoer, huurauto's of taxi's).

- Accommodatie:
Hotelovernachting voor drie nachten in een middenklasse zakenhotel dichtbij de vergaderlocatie.

- Maaltijden:
Dagelijkse maaltijdvergoeding voor ontbijt, lunch en diner, gebaseerd op het dagvergoedingbeleid van het bedrijf.

- Diverse onkosten:
Parkeerbetalingen bij de luchthaven.
Internettoegangskosten in het hotel.
Fooien of kleine servicekosten.

- Documentatie:
Je dient alle bonnetjes in (vluchten, taxi’s, hotel, maaltijden, enz.) en een ingevuld onkostennotaformulier voor vergoeding.


## Vereiste bibliotheken importeren

Importeer de benodigde bibliotheken en modules voor het notitieboek.


In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os
import base64
import dotenv
from typing import Annotated, List

from pydantic import BaseModel, Field

from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

 ## Definieer kostenmodellen

 Maak een Pydantic-model voor individuele kosten en een ExpenseFormatter-klasse om een gebruikersquery om te zetten in gestructureerde kosteninformatie.

 Elke kost wordt weergegeven in het formaat:
 `{'date': '07-Mar-2025', 'description': 'flight to destination', 'amount': 675.99, 'category': 'Transportation'}`


In [ ]:
class Expense(BaseModel):
    date: str = Field(..., description="Date of expense in dd-MMM-yyyy format")
    description: str = Field(..., description="Expense description")
    amount: float = Field(..., description="Expense amount")
    category: str = Field(..., description="Expense category (e.g., Transportation, Meals, Accommodation, Miscellaneous)")

class ExpenseFormatter(BaseModel):
    raw_query: str = Field(..., description="Raw query input containing expense details")
    
    def parse_expenses(self) -> List[Expense]:
        """
        Parses the raw query into a list of Expense objects.
        Expected format: "date|description|amount|category" separated by semicolons.
        """
        expense_list = []
        for expense_str in self.raw_query.split(";"):
            if expense_str.strip():
                parts = expense_str.strip().split("|")
                if len(parts) == 4:
                    date, description, amount, category = parts
                    try:
                        expense = Expense(
                            date=date.strip(),
                            description=description.strip(),
                            amount=float(amount.strip()),
                            category=category.strip()
                        )
                        expense_list.append(expense)
                    except ValueError as e:
                        print(f"[LOG] Parse Error: Invalid data in '{expense_str}': {e}")
        return expense_list

## Definiëren van Hulpmiddelen - Het genereren van de e-mail

Maak een hulpmiddel-functie om een e-mail te genereren voor het indienen van een onkostennota.
- Dit hulpmiddel gebruikt de `@tool` decorator van het Microsoft Agent Framework.
- Het berekent het totale bedrag van de onkosten en formatteert de details in een e-mailtekst.


In [ ]:
@tool(approval_mode="never_require")
def generate_expense_email(
    expense_data: Annotated[str, "Semicolon-separated expense entries in 'date|description|amount|category' format"]
) -> str:
    """Generate an email to submit an expense claim to the Finance Team."""
    formatter = ExpenseFormatter(raw_query=expense_data)
    expenses = formatter.parse_expenses()
    if not expenses:
        return "No valid expenses found to include in the email."
    total_amount = sum(e.amount for e in expenses)
    email_body = "Dear Finance Team,\n\n"
    email_body += "Please find below the details of my expense claim:\n\n"
    for e in expenses:
        email_body += f"- {e.date} | {e.description}: ${e.amount:.2f} ({e.category})\n"
    email_body += f"\nTotal Amount: ${total_amount:.2f}\n\n"
    email_body += "Receipts for all expenses are attached for your reference.\n\n"
    email_body += "Thank you,\n[Your Name]"
    return email_body

# Hulpmiddel voor het extraheren van reiskosten uit bonafbeeldingen

Maak een hulpprogrammafunctie om reiskosten uit bonafbeeldingen te extraheren.
- Dit hulpmiddel gebruikt de `@tool` decorator van het Microsoft Agent Framework.
- Het leest de bonafbeelding, codeert deze als base64 en retourneert de data-URI voor de agent om te analyseren.


In [ ]:
@tool(approval_mode="never_require")
def load_receipt_image(
    image_path: Annotated[str, "Path to the receipt image file"] = "receipt.jpg"
) -> str:
    """Load a receipt image and return its base64-encoded data URI for OCR extraction."""
    try:
        with open(image_path, "rb") as f:
            image_data = base64.b64encode(f.read()).decode("utf-8")
        return f"data:image/jpeg;base64,{image_data}"
    except Exception as e:
        error_msg = f"[LOG] Error loading image '{image_path}': {str(e)}"
        print(error_msg)
        return error_msg

## Verwerking van onkosten

Definieer de agenten en verbind ze in een sequentiële workflow met behulp van `WorkflowBuilder`.
- De OCR-agent extraheert gestructureerde onkostengegevens uit de bonafbeelding met de tool `load_receipt_image`.
- De Email-agent neemt de geëxtraheerde gegevens en genereert een professionele onkostendeclaratiemail met de tool `generate_expense_email`.
- `WorkflowBuilder` met `add_edge` creëert een sequentiële pijplijn: OCR-agent → Email-agent.


In [ ]:
ocr_agent = client.as_agent(
    tools=[load_receipt_image],
    name="OCRAgent",
    instructions=(
        "You are an expert OCR assistant specialized in extracting structured data from receipt images. "
        "Use the 'load_receipt_image' tool to load the receipt image, then analyze it and extract "
        "travel-related expense details in the format: 'date|description|amount|category' separated by semicolons. "
        "Follow these rules: "
        "- Date: Convert dates (e.g., '4/4/22') to 'dd-MMM-yyyy' (e.g., '04-Apr-2022'). "
        "- Description: Extract item names. "
        "- Amount: Use numeric values (e.g., '4.50' from '$4.50'). "
        "- Category: Infer from context (e.g., 'Meals' for food, 'Transportation' for travel, "
        "'Accommodation' for lodging, 'Miscellaneous' otherwise). "
        "Ignore totals, subtotals, or service charges unless they are itemized expenses. "
        "If no expenses are found, return 'No expenses detected'. "
        "Return only the structured data, no additional text."
    ),
)

email_agent = client.as_agent(
    name="EmailAgent",
    tools=[generate_expense_email],
    instructions=(
        "You are an expense claim email generator. Take the travel expense data from the previous agent "
        "(in 'date|description|amount|category' format separated by semicolons) and use the "
        "'generate_expense_email' tool to produce a professional expense claim email. "
        "Pass the semicolon-separated expense data directly to the tool."
    ),
)

## Hoofdfunctie

Bouw de sequentiële workflow en voer deze uit om de bonafbeelding te verwerken en de onkostendeclaratiemail te genereren.


> **Opmerking:** Deze workflow geeft momenteel de bonafbeelding door als base64-tekst, die de meeste chatmodellen (inclusief gpt-4.1-mini) niet als een afbeelding zullen behandelen.
> Het kan ook de contextvenster van het model overschrijden. Voer bij voorkeur OCR uit met Azure AI Vision (of een andere OCR-tool) en geef alleen de geëxtraheerde tekst door, of herstructureer om de afbeelding als een `image_url`-bericht te verzenden.
> Als je alleen contextfouten wilt vermijden, probeer dan een kleinere bonafbeelding of een model met een groter contextvenster.


In [ ]:
workflow = WorkflowBuilder(start_executor=ocr_agent) \
    .add_edge(ocr_agent, email_agent) \
    .build()

prompt = (
    "Please extract the raw text from the receipt image at 'receipt.jpg', "
    "focusing on travel expenses like dates, descriptions, amounts, and categories "
    "(e.g., Transportation, Accommodation, Meals, Miscellaneous). "
    "Then generate a professional expense claim email."
)

last_author = None
events = workflow.run(
    prompt,
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"# Agent - {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Dit document is vertaald met behulp van de AI vertaaldienst [Co-op Translator](https://github.com/Azure/co-op-translator). Hoewel we streven naar nauwkeurigheid, dient u er rekening mee te houden dat geautomatiseerde vertalingen fouten of onnauwkeurigheden kunnen bevatten. Het originele document in de oorspronkelijke taal moet worden beschouwd als de gezaghebbende bron. Voor kritieke informatie wordt professionele menselijke vertaling aanbevolen. Wij zijn niet aansprakelijk voor eventuele misverstanden of verkeerde interpretaties die voortvloeien uit het gebruik van deze vertaling.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
